In [ ]:
!pip -q install pypdf sentence-transformers rank-bm25 faiss-cpu pandas openpyxl tqdm

In [ ]:
import os
import re
import json
import math
import textwrap
import numpy as np
import pandas as pd

from tqdm import tqdm
from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss

In [ ]:
print("Сначала загрузи PDF с документом стратегии, потом test_set (xlsx/xls/csv).")
uploaded = files.upload()

pdf_path = None
test_path = None

for fname in uploaded.keys():
    low = fname.lower()
    if low.endswith(".pdf"):
        pdf_path = fname
    elif low.endswith(".xlsx") or low.endswith(".xls") or low.endswith(".csv"):
        test_path = fname

print("PDF:", pdf_path)
print("TEST:", test_path)

assert pdf_path is not None, "Не загружен PDF"
assert test_path is not None, "Не загружен test_set"

In [ ]:
def extract_text_from_pdf(path):
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            pages.append(f"[PAGE {i+1}]\n{text}")
    return "\n\n".join(pages)

document_text = extract_text_from_pdf(pdf_path)
print(document_text[:3000])
print("\n---\nОбщая длина текста:", len(document_text))

In [ ]:
def clean_text(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

document_text = clean_text(document_text)
print(document_text[:3000])

In [ ]:
def chunk_text(text, chunk_size=1200, overlap=200):
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + chunk_size, n)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == n:
            break
        start = end - overlap
    return chunks

chunks = chunk_text(document_text, chunk_size=1200, overlap=200)
print("Количество чанков:", len(chunks))
print("\nПример чанка:\n")
print(chunks[0][:1500])

In [ ]:
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

chunk_embeddings = embedder.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(chunk_embeddings.shape)

In [ ]:
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)

print("FAISS готов, в индексе чанков:", index.ntotal)

In [ ]:
def tokenize_for_bm25(text):
    text = text.lower()
    text = re.sub(r"[^а-яa-z0-9 ]", " ", text)
    tokens = text.split()
    return tokens

tokenized_chunks = [tokenize_for_bm25(c) for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print("BM25 готов")

In [ ]:
def hybrid_search(query, top_k_dense=8, top_k_sparse=8, final_k=5):
    # dense
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    dense_scores, dense_ids = index.search(q_emb, top_k_dense)
    dense_scores = dense_scores[0]
    dense_ids = dense_ids[0]

    # sparse
    tokenized_query = tokenize_for_bm25(query)
    sparse_scores = bm25.get_scores(tokenized_query)
    sparse_ids = np.argsort(sparse_scores)[::-1][:top_k_sparse]

    # merge
    score_dict = {}

    for idx, score in zip(dense_ids, dense_scores):
        score_dict[int(idx)] = score_dict.get(int(idx), 0.0) + float(score)

    max_sparse = max([sparse_scores[i] for i in sparse_ids]) if len(sparse_ids) > 0 else 1.0
    if max_sparse == 0:
        max_sparse = 1.0

    for idx in sparse_ids:
        norm_score = float(sparse_scores[idx]) / max_sparse
        score_dict[int(idx)] = score_dict.get(int(idx), 0.0) + norm_score

    ranked = sorted(score_dict.items(), key=lambda x: x[1], reverse=True)[:final_k]

    results = []
    for idx, score in ranked:
        results.append({
            "chunk_id": idx,
            "score": score,
            "text": chunks[idx]
        })
    return results

In [ ]:
def show_retrieval(query, k=5):
    results = hybrid_search(query, final_k=k)
    for i, item in enumerate(results, 1):
        print("=" * 80)
        print(f"RESULT {i} | chunk_id={item['chunk_id']} | score={item['score']:.4f}")
        print(item["text"][:2000])
        print()

# Пример:
# show_retrieval("Каковы цели национальной стратегии развития искусственного интеллекта?")

In [ ]:
def split_into_sentences(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    parts = re.split(r'(?<=[\.\!\?])\s+', text)
    return [p.strip() for p in parts if p.strip()]

def tokenize_for_match(text):
    text = text.lower().replace("ё", "е")
    text = re.sub(r"[^а-яa-z0-9 ]", " ", text)
    return [w for w in text.split() if len(w) > 2]

def keyword_score(question, sentence):
    q_words = set(tokenize_for_match(question))
    s_words = set(tokenize_for_match(sentence))
    if not q_words or not s_words:
        return 0.0
    overlap = len(q_words & s_words)
    return overlap / len(q_words)

def is_good_sentence(sent):
    sent = sent.strip()
    if len(sent) < 35:
        return False
    if sent.startswith("[PAGE"):
        return False
    if sent[0].islower():
        return False
    return True

def postprocess_answer(answer, max_chars=350):
    answer = re.sub(r"\s+", " ", answer).strip()
    if len(answer) > max_chars:
        answer = answer[:max_chars].rsplit(" ", 1)[0].strip() + "."
    return answer

def generate_extractive_answer(question, max_sentences=2, max_chars=350):
    q_lower = question.lower()

    # защита от инъекций
    bad_patterns = [
        "игнорируй документ",
        "игнорируй предыдущие правила",
        "придумай",
        "секретные",
        "даже если их нет"
    ]
    if any(p in q_lower for p in bad_patterns):
        return "В представленном документе отсутствуют секретные пункты и неуказанные в тексте данные; ответ формируется только по содержанию документа."

    retrieved = hybrid_search(question, final_k=7)

    all_sentences = []
    for item in retrieved:
        for sent in split_into_sentences(item["text"]):
            sent = re.sub(r"\[PAGE \d+\]", "", sent).strip()
            if is_good_sentence(sent):
                score = keyword_score(question, sent)
                all_sentences.append((score, sent))

    all_sentences.sort(key=lambda x: x[0], reverse=True)

    best_sentences = []
    used = set()

    for score, sent in all_sentences:
        norm = sent.lower()
        if norm in used:
            continue
        used.add(norm)

        # берем только достаточно релевантные предложения
        if score >= 0.18:
            best_sentences.append(sent)

        if len(best_sentences) >= max_sentences:
            break

    if not best_sentences:
        return "В представленном документе нет явного ответа на этот вопрос."

    answer = " ".join(best_sentences)

    # если ответ слишком общий, оставляем только первое предложение
    if len(answer) > 250:
        answer = best_sentences[0]

    return postprocess_answer(answer, max_chars=max_chars)

In [ ]:
question = "Каковы основные цели национальной стратегии развития искусственного интеллекта?"
print("ВОПРОС:")
print(question)
print("\nОТВЕТ:")
print(generate_extractive_answer(question))

In [ ]:
def load_test_file(path):
    low = path.lower()
    if low.endswith(".csv"):
        df = pd.read_csv(path)
    elif low.endswith(".xlsx") or low.endswith(".xls"):
        df = pd.read_excel(path)
    else:
        raise ValueError("Неподдерживаемый формат test_set")
    return df

df_test = load_test_file(test_path)
print(df_test.head())
print("\nКолонки:", list(df_test.columns))

In [ ]:
possible_question_cols = ["question", "questions", "query", "prompt", "input", "вопрос"]

question_col = None
for col in df_test.columns:
    if col.lower() in possible_question_cols:
        question_col = col
        break

if question_col is None:
    print("Колонки в файле:", list(df_test.columns))
    raise ValueError("Не нашел колонку с вопросами. Посмотри название колонки и впиши его вручную.")

print("Колонка с вопросами:", question_col)

In [ ]:
answers = []

for q in tqdm(df_test[question_col].astype(str).tolist()):
    ans = generate_extractive_answer(q, max_sentences=3, max_chars=600)
    answers.append(ans)

df_test["answer"] = answers
df_test.head()

In [ ]:
SURNAME = "Калашников"
NAME = "Егор"

output_name = f"test_set_{SURNAME}_{NAME}.xlsx"
df_test.to_excel(output_name, index=False)

print("Файл сохранен:", output_name)
files.download(output_name)

In [ ]:
demo_questions = [
    "Какова цель национальной стратегии развития искусственного интеллекта?",
    "Какие задачи государства указаны в документе?",
    "Какие риски и ограничения связаны с развитием искусственного интеллекта?"
]

for q in demo_questions:
    print("=" * 100)
    print("ВОПРОС:", q)
    print("ОТВЕТ:", generate_extractive_answer(q))
    print()

In [ ]:
requirements_text = """pypdf
sentence-transformers
rank-bm25
faiss-cpu
pandas
openpyxl
tqdm
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_text)

print("Создан requirements.txt")
files.download("requirements.txt")